# SHiP adaptive muon background — Colab quickstart

Clone → install → call. This notebook **defines no functions**: all logic
lives in the installed packages (`ship_muon_bg`, `Nflow`, `ProxyTagger`).
It exercises the tested pieces of the architecture end to end on the tiny
committed fixture — no GPU, no FairShip, no ROOT needed.

In [ ]:
!git clone https://github.com/fbientrigo/ship_adaptive_muon_bg.git
%cd ship_adaptive_muon_bg
# While the architecture branch is under review; drop after merge to main:
!git checkout feat/architecture
%pip install --quiet .

## 1. Data contract: load, validate, hash, split, normalize

`process_pkl` runs the full v0 contract on a muon PKL file and returns the
three provenance artifacts (`dataset_report`, `split_manifest`,
`normalization`).

In [ ]:
from ship_muon_bg.data_contracts import process_pkl

artifacts = process_pkl(
    "tests/fixtures/muon_sample_tiny.pkl.gz", seed=1234, val_fraction=0.25
)
report = artifacts["dataset_report"]
print("rows:", report["n_rows"])
print("dataset_hash:", report["dataset_hash"][:16], "...")
print("train/val:", len(artifacts["split_manifest"]["train_indices"]),
      "/", len(artifacts["split_manifest"]["val_indices"]))

## 2. ProxyTagger: score candidates with the (non-physical) `DummyProxy`

`U(x)` semantics: a continuous worth-simulating score in [0, 1]. The dummy
baseline only exercises the interface — `is_physical=False`, never a result.

In [ ]:
from ship_muon_bg.data_contracts import load_muon_pkl
from ProxyTagger import DummyProxy

muons = load_muon_pkl("tests/fixtures/muon_sample_tiny.pkl.gz")
proxy = DummyProxy()
scores = proxy.score(muons)
print(proxy.name, "| physical:", proxy.is_physical)
print("U(x) range on fixture:", scores.min(), "-", scores.max())

## 3. The three module interfaces

Everything future work plugs into: proposal models behind `DensityModel`,
biasing mechanisms behind `BiasStrategy` (the A/B seam), proxies behind
`ProxyScorer`, and simulators behind `SimulationBackend`.

In [ ]:
from Nflow import BiasStrategy, DensityModel
from ProxyTagger import ProxyScorer
from ship_muon_bg.simulation import (
    FlowProposalRecord,
    OutcomeCategory,
    SimulationBackend,
    SimulationResult,
)

for protocol in (DensityModel, BiasStrategy, ProxyScorer, SimulationBackend):
    print(f"== {protocol.__name__} ==")
    print(protocol.__doc__.strip().splitlines()[0], "\n")

print("Outcome taxonomy:", [c.value for c in OutcomeCategory])

## Where to go next

See `docs/architecture/repo_architecture_v1.md` for the module map and the
ordered best-ROI next steps (real flow behind `DensityModel`,
`toy_simulator` for first labels, `fairship_adapter` dry-run, A/B bias
harness, CI).